# Solaris

## Imports

In [8]:
from contextlib import contextmanager
from dataclasses import dataclass
import matplotlib.pyplot as plt
import gymnasium as gym
import numpy as np
import ale_py
import pygame

## Environment

In [ ]:
# I will use context managers for managing the environment to ensure that it will be reset and closed
WINDOW_SIZE = (455, 552)

@contextmanager
def create_env(env_name, live_rendering=False, **kwargs):
    env = gym.make(env_name, **kwargs)
    env.__setattr__('_live_rendering', live_rendering)

    if live_rendering:
        if env.render_mode != 'rgb_array':
            raise ValueError('Render mode should be rgb_array for live rendering')
        
        pygame.init()
        pygame.display.set_caption(env_name)
        pygame.event.set_allowed([pygame.QUIT, pygame.KEYDOWN])

        env.__setattr__('_clock', pygame.time.Clock())
        env.__setattr__('_screen', pygame.display.set_mode(WINDOW_SIZE, pygame.RESIZABLE))
    try:
        yield env
    finally:
        env.close()

        if live_rendering:
            pygame.quit()

# I will also use a dataclass for episode to keep the loop cleaner
@dataclass
class Episode:
    env: gym.Env
    
    obs: object = None
    reward: float | None = None
    terminated: bool = False
    truncated: bool = False
    info: dict | None = None

    stop_rendering = False

    def _render(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.stop_rendering = True

                return

        screen = self.env.__getattribute__('_screen')
        # We will remove the black left border
        frame = self.env.render()[:, 8:, :]
        
        raw_surface = pygame.surfarray.make_surface(np.transpose(frame, (1, 0, 2)))
        upscaled_surface = pygame.transform.scale_by(raw_surface, 3)

        scr_w, scr_h = screen.get_size()
        surf_w, surf_h = upscaled_surface.get_size()
        
        x_offset = (scr_w - surf_w) // 2
        y_offset = (scr_h - surf_h) // 2

        screen.fill((0, 0, 0))
        screen.blit(upscaled_surface, (x_offset, y_offset))

        pygame.display.flip()

        self.env.__getattribute__('_clock').tick(60)

    @property
    def done(self):
        return self.terminated or self.truncated or self.stop_rendering
    
    def step(self, action):
        self.obs, self.reward, self.terminated, self.truncated, self.info = self.env.step(action)

        if not self.done and self.env.__getattribute__('_live_rendering'):
            self._render()

@contextmanager
def create_episode(env, **kwargs):
    obs, info = env.reset(**kwargs)

    yield Episode(env, obs=obs, info=info)

In [104]:
with create_env('ALE/Solaris-v5', live_rendering=True, render_mode='rgb_array') as env:
    for _ in range(10):
        with create_episode(env) as episode:
            while not episode.done:
                action = env.action_space.sample()
                episode.step(action)

            if episode.stop_rendering:
                break